# Agente investigador de incidentes de produção

Projeto do item 1 da atividade de **AI Agents com LangChain**: da ideia ao MVP.
O agente conversa com o engenheiro de plantão, responde sobre um incidente **com evidência citada**
e recusa o que está fora do escopo. Ele só lê e sugere; nunca executa ações em produção.

Este notebook é único e cresce por seção, uma por fase. Nesta versão:

- **Seção 1 · Fase 1**: chain executável com entrada e saída estruturadas (MVP, sem ferramentas e sem memória de longo prazo).

O código reutilizável mora no pacote `agente/` (testado com `pytest`); o notebook importa e exibe o que é avaliado com `inspect.getsource`.
Desenho completo do projeto: [artefato do Projeto](https://claude.ai/artifact/MgdyrpHc8ZPoP7CryN9tRu).

## Bootstrap

Coloca a raiz do repositório no `sys.path`. Se o notebook rodar isolado (ex.: Colab) e a pasta `agente/` não existir, clona o repositório público.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

raiz = Path.cwd()
if not (raiz / "agente").exists():
    if (raiz.parent / "agente").exists():
        raiz = raiz.parent
    else:
        subprocess.run(["git", "clone", "https://github.com/erato690/incident-agent"], check=True)
        raiz = (raiz / "incident-agent").resolve()
os.chdir(raiz)
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))
print("Raiz do projeto:", raiz)

## Setup

Este notebook usa o **plano B** (OpenAI, `gpt-4o-mini`). Configure o `.env` na raiz do repositório (ou as variáveis de ambiente, no Colab):

```
OPENAI_API_KEY=sk-...     # usada pelo LLM e também pela moderação gratuita
FORCAR_PLANO_B=1          # força o gpt-4o-mini mesmo que exista DEEPSEEK_API_KEY
```

Sem `OPENAI_API_KEY` o LLM não é criado (erro explicando o que definir) e a moderação seria pulada com aviso. No Colab, descomente a linha do `pip` abaixo.

In [ ]:
# %pip install -q -r requirements.txt   # descomente no Colab

# .env esperado: OPENAI_API_KEY=... e FORCAR_PLANO_B=1 (plano B: gpt-4o-mini)
from agente.config import escolher_llm
from agente.moderacao import moderar

llm = escolher_llm()  # imprime o modelo escolhido

---
# Seção 1 · Fase 1: chain executável com entrada e saída estruturadas

Pipeline do MVP, tudo em LCEL:

`mensagem → máscara local → moderação → NLU → roteamento → (resposta fixa | pré-processamento | prompt | modelo | parser JSON)`

Cada subseção abaixo comprova um item da rubrica; a tabela da **1.12** liga item e célula.

## 1.1 Anatomia, tipo e maturidade agêntica

Resumo do desenho (aba *Projeto* do [artefato](https://claude.ai/artifact/MgdyrpHc8ZPoP7CryN9tRu)). A anatomia tem **7 etapas numeradas**:

1. **Moderação**: endpoint gratuito da OpenAI; mensagem sinalizada recebe resposta fixa, sem chamar o agente.
2. **Barreira + NLU**: intenção e entidades com o histórico; saudação ou fora do escopo recebe resposta fixa.
3. **Agente**: LLM com system message (PTCF) decompõe a pergunta e decide o que consultar.
4. **Ferramentas**: só leitura em logs, métricas, deploys, runbooks e GitHub (fase 2).
5. **Memória**: lê curto e longo prazo; grava só o que o engenheiro confirmou (fases 3 e 4).
6. **Auditor** (fase 5): confere a resposta contra a evidência antes de entregar.
7. **Post-mortem `.md`, só no final**: único atuador; sai quando o engenheiro pede ao fim da investigação.

Os quatro elementos de um agente: ambiente (produção), sensores (ferramentas de leitura), cognição (LLM com system message) e atuador (post-mortem `.md`).

**Tipo (padrão arquitetural): híbrido.** Reativo para disparar (cada pergunta do engenheiro aciona um turno; a barreira reage na hora a saudações e assuntos alheios) e deliberativo para investigar (monta hipóteses, escolhe a próxima consulta, confronta evidência), pendendo ao lado deliberativo. Quanto ao tipo, é mistura de **tarefa** (executa a investigação com ferramentas) e **busca agêntica** (consulta runbooks e incidentes antigos em várias etapas). Autônomo não é a meta: o humano decide.

**Maturidade agêntica, de 0 a 4:**

| Nível | Descrição | Situação |
|---|---|---|
| 0 · LLM isolado | Responde ao que foi perguntado, sem contexto nem dados externos | |
| **1 · Chain fixa** | Prompt, modelo e parser; fluxo decidido no código | **hoje (esta fase)** |
| 2 · Agente com ferramentas | Ciclo ReAct decide quando consultar logs, deploys e runbooks | fase 2 |
| 3 · Com memória | Lembra do incidente atual e de incidentes passados | fases 3 e 4 |
| 4 · Com verificação | Auditor de verificação cognitiva checa cada resposta contra a evidência | meta, fase 5 |

## 1.2 Planejamento hierárquico e memória aumentada

**Planejamento hierárquico.** Objetivo: *achar a causa raiz do erro*, em 5 etapas numeradas, porque seguem uma ordem real:

1. **Entender o erro**: extrair exceção e trace, achar serviço e versão, classificar o tipo. *(MVP)*
2. **Delimitar**: quando começou, o que foi afetado, definir severidade. *(MVP)*
3. **Correlacionar**: deploys recentes e picos de métricas; a causa vem antes do efeito. *(fase 2, ferramentas)*
4. **Hipóteses**: listar causas possíveis, ranquear por confiança, casos parecidos (fase 3). *(MVP, só com o texto)*
5. **Verificar**: evidência a favor e contra cada hipótese, checar com o runbook. *(fases 4 e 5)*

Nesta fase só as etapas 1, 2 e 4 funcionam, e apenas com o texto do erro; sem dado suficiente a análise diz "evidência insuficiente".

**Memória aumentada (CoALA).** Quatro memórias:

| Memória | No domínio de incidentes | Fase |
|---|---|---|
| Trabalho (curto prazo) | Conversa do incidente, entidades acumuladas, saídas das ferramentas (`MemorySaver` com `thread_id` por incidente) | 3 |
| Episódica | Incidentes passados: impressão digital, sintoma, causa confirmada, correção, data | 3 e 4 |
| Semântica | Mapa de dependências, runbooks, limites e configurações (FAISS) | 4 |
| Procedural | System message e regras de investigação (prompt versionado no repositório) | 1; escrita no fim |

O agente **grava só o que o engenheiro confirmou** (intenção `confirmar_causa`); hipótese do agente não vira fato. Nesta fase o histórico é só uma lista de mensagens passada explicitamente.

## 1.3 Templates reutilizáveis: `PromptTemplate` e `ChatPromptTemplate`

Definidos uma vez em `agente/prompts.py` e reutilizados pelo pipeline: `PromptTemplate` nas respostas fixas, `ChatPromptTemplate` no NLU e na investigação.

In [ ]:
import inspect
from agente import prompts

print(inspect.getsource(prompts))

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from agente.prompts import prompt_investigacao, prompt_nlu, respostas_fixas

# PromptTemplate: texto simples
print("PromptTemplate.format():")
print(respostas_fixas["fora_do_escopo"].format())

# ChatPromptTemplate: mensagens, com o histórico no MessagesPlaceholder
historico = [HumanMessage("O pagamentos-api está com 500."), AIMessage("Qual o texto do erro?")]
print("\nChatPromptTemplate.format_messages():")
for m in prompt_nlu.format_messages(historico=historico, mensagem="foi o deploy?"):
    print(f"[{m.type}] {m.content[:90]!r}")

## 1.4 Máscara, pré-processamento no pipe e `@chain`

Antes de qualquer envio externo, a máscara local (sem LLM) troca segredos e dados pessoais por marcadores. Em seguida o pré-processamento limpa o ruído do stack trace e extrai a **impressão digital** do erro (exceção, primeiro frame da aplicação, serviço).

In [ ]:
import json
from agente.mascara import mascarar

mensagens = json.loads((raiz / "dados_dummy" / "fase1_mensagens.json").read_text(encoding="utf-8"))
por_id = {m["id"]: m for m in mensagens}

bruto = por_id["com_segredos"]["mensagem"]
print("Original :", bruto)
print("Mascarado:", mascarar(bruto))

In [ ]:
from langchain_core.runnables import RunnableLambda
from agente.preprocessamento import preprocessar

print(inspect.getsource(preprocessar))

# No pipeline, o passo entra como função comum composta no pipe: RunnableLambda(preprocessar)
trace = por_id["deploy_com_bug"]["mensagem"]
saida_pipe = RunnableLambda(preprocessar).invoke(trace)
print("Impressão digital:", saida_pipe.impressao)
print("Texto limpo:\n", saida_pipe.texto_limpo)

**O mesmo passo com `@chain`** (composição imperativa): o decorador transforma a função em `RunnableLambda`.

In [ ]:
from langchain_core.runnables import chain, RunnableLambda

@chain
def preprocessar_chain(texto):
    return preprocessar(texto)

saida_chain = preprocessar_chain.invoke(trace)
print(RunnableLambda(preprocessar).invoke(trace) == preprocessar_chain.invoke(trace))  # True
print(type(preprocessar_chain).__name__)

print("Equivalência: `@chain` e `RunnableLambda(função)` produzem o mesmo objeto, um RunnableLambda;")
print("o projeto usa a função no pipe porque a composição LCEL (`|`) fica explícita e visível no pipeline.")

## 1.5 Saída estruturada em JSON com output parser

A investigação devolve um JSON validado pelo schema `AnaliseIncidente`: `JsonOutputParser` (que também faz *streaming* de JSON parcial) seguido de `AnaliseIncidente.model_validate`, com **uma** nova tentativa (`with_retry`). Se falhar de novo, devolve um erro estruturado sem quebrar o resto.

In [ ]:
from agente.pipeline import montar_investigacao
from agente.schemas import AnaliseIncidente, Hipotese

print(inspect.getsource(Hipotese))
print(inspect.getsource(AnaliseIncidente))
print(inspect.getsource(montar_investigacao))

In [ ]:
# Validação do schema sem LLM: ordena hipóteses por confiança e exige 'evidência insuficiente' quando não há hipótese
exemplo = AnaliseIncidente.model_validate({
    "severidade": "alta", "categoria": "nullpointer", "servico": "pagamentos-api",
    "resumo": "NPE ao confirmar pagamento após o deploy.",
    "hipoteses": [
        {"causa": "cupom nulo não tratado", "confianca": 0.4, "evidencia": "cupom is null"},
        {"causa": "regressão do deploy v2.14.0", "confianca": 0.8, "evidencia": "depois do deploy v2.14.0"},
    ],
    "proximos_passos": ["Ver o diff do deploy v2.14.0 em PagamentoService.confirmar"],
})
print([(h.causa, h.confianca) for h in exemplo.hipoteses])
try:
    AnaliseIncidente.model_validate({"severidade": "alta", "categoria": "x", "resumo": "r",
                                     "hipoteses": [], "proximos_passos": ["p"]})
except Exception as e:
    print("Rejeitado como esperado:", e.errors()[0]["msg"][:90])

## 1.6 Chain completa em LCEL

`montar_pipeline` compõe tudo de forma **declarativa**: máscara → `RunnablePassthrough.assign` (moderação, NLU) → `RunnableBranch` (resposta fixa ou investigação). Moderação sinalizada sai antes do NLU; intenções de guardrail, saudação, fora do escopo, `gerar_relatorio` e `confirmar_causa` recebem resposta fixa; o resto vai para a investigação.

In [ ]:
from agente.pipeline import montar_pipeline

print(inspect.getsource(montar_pipeline))

pipeline = montar_pipeline(llm, moderar)
print(type(pipeline).__name__)

## 1.7 Interface Runnable: `invoke`

Uma mensagem (cenário `deploy_com_bug`) pela chain completa.

In [ ]:
entrada = {"mensagem": por_id["deploy_com_bug"]["mensagem"], "historico": []}
resp = pipeline.invoke(entrada)

print("Rota:", resp.rota, "| Moderação:", resp.moderacao.status, "| Intenção:", resp.nlu.intencao)
print(resp.analise.model_dump_json(indent=2) if resp.analise else resp.erro)

## 1.8 Interface Runnable: `batch`

As 8 mensagens dos cenários dummy em uma chamada.

In [ ]:
cenarios = [m for m in mensagens if m["rota_esperada"] == "investigacao"][:8]
respostas = pipeline.batch([{"mensagem": m["mensagem"], "historico": []} for m in cenarios])

print(f"{'cenário':<24} {'severidade':<11} primeira hipótese")
for m, r in zip(cenarios, respostas):
    if r.analise and r.analise.hipoteses:
        sev, hip = r.analise.severidade, r.analise.hipoteses[0].causa
    else:
        sev, hip = "-", r.erro or "(sem hipótese)"
    print(f"{m['id']:<24} {sev:<11} {hip[:70]}")

## 1.9 Interface Runnable: `stream`

O parser JSON emite o objeto **crescendo** à medida que o modelo gera os tokens. A validação final (`model_validate`) precisa do JSON completo, por isso o `stream` abaixo usa a investigação até o parser, `montar_investigacao_json`, que é a mesma investigação sem a validação final (pré-processamento, prompt, modelo e `JsonOutputParser`).

In [ ]:
from agente.pipeline import montar_investigacao_json

# Mesmas peças da investigação, até o JsonOutputParser (sem a validação final).
investigacao_stream = montar_investigacao_json(llm)

estado = {"mensagem": mascarar(por_id["pool_esgotado"]["mensagem"])}
parcial = {}
for i, parcial in enumerate(investigacao_stream.stream(estado), start=1):
    if i % 5 == 1:  # mostra 1 de cada 5 pedaços para não poluir
        print(f"pedaço {i:>3}:", json.dumps(parcial, ensure_ascii=False)[:120])
print(f"\nTotal de pedaços: {i}")
print("Final válido no schema:", AnaliseIncidente.model_validate(parcial).severidade)

## 1.10 Bateria de mensagens de teste

Roda `dados_dummy/fase1_mensagens.json` (cenários, saudação, fora do escopo, pedido de ação, injeção, sintoma sem erro, OOMKilled, mensagem mista, vazia e com segredos) e compara a rota esperada com a obtida. Depois verifica que nenhum valor original de segredo aparece no texto mascarado nem na análise final.

In [ ]:
resultados = pipeline.batch([{"mensagem": m["mensagem"], "historico": []} for m in mensagens])

def intencao_obtida(r):
    if r.nlu:
        return r.nlu.intencao
    return r.fixa.categoria if r.fixa else "-"

print(f"{'id':<24} {'esperada':<13} {'obtida':<13} {'intenção':<22} ok")
acertos = 0
for m, r in zip(mensagens, resultados):
    ok = r.rota == m["rota_esperada"]
    acertos += ok
    print(f"{m['id']:<24} {m['rota_esperada']:<13} {r.rota:<13} {intencao_obtida(r):<22} {'ok' if ok else 'FALHOU'}")
print(f"\n{acertos}/{len(mensagens)} rotas corretas")

In [ ]:
# Segredos da mensagem 'com_segredos' não podem aparecer no texto mascarado nem na análise final
segredos = ["Sup3rS3cret", "abc123def456", "ana.souza@loja.com.br"]
mascarado = mascarar(por_id["com_segredos"]["mensagem"])
r_seg = resultados[[m["id"] for m in mensagens].index("com_segredos")]
saida_final = r_seg.analise.model_dump_json() if r_seg.analise else (r_seg.erro or "")

print("Mascarado:", mascarado)
print("Nenhum segredo no texto mascarado:", not any(s in mascarado for s in segredos))
print("Nenhum segredo na análise final  :", not any(s in saida_final for s in segredos))

## 1.11 LCEL × composição imperativa

Composição imperativa (`@chain`) × declarativa (LCEL), aplicada ao projeto:

| Característica | Imperativa (`@chain`) | Declarativa (LCEL) |
|---|---|---|
| Definição | Dita o "como", passo a passo, gerenciando o estado manualmente | Declara o "o quê": a topologia; o framework gerencia o fluxo de dados |
| Sintaxe | Função Python decorada com `@chain` | Operador pipe `A \| B` |
| Objeto resultante | `RunnableLambda` | `RunnableSequence` |
| Streaming e batch | Executa linha por linha; sem otimização automática | `batch` paralelo e `stream` automático entre as peças |
| Vantagem | Depuração com `print`, `if/else/for` nativos | Conciso, interface unificada, grafo legível |
| Desvantagem | Mais verboso; perde otimizações | Fluxo oculto no `\|` dificulta inspecionar etapas |

**Roteamento.** Uma alternativa é rotear com uma função Python dentro de um `RunnableLambda` que *devolve a chain de destino*. Este projeto declara as rotas com `RunnableBranch`, uma lista de pares (condição, destino) que continua sendo LCEL e cabe no desenho do Projeto: a decisão (`intencao in INTENCOES_FIXAS`) fica visível na composição, e a moderação sinalizada sai por um segundo `RunnableBranch` no topo.

**Escolha justificada.** O pipeline é **declarativo (LCEL)** porque é uma topologia fixa (máscara, moderação, NLU, roteamento, investigação) e ganhamos `invoke`, `batch` e `stream` sem código extra, com resultados tipados entre as etapas. Cada etapa determinística (máscara, pré-processamento) é uma função comum testada com `pytest` e entra no pipe como `RunnableLambda`. Usamos `@chain` só onde a lógica imperativa compensa, e aqui ele é apenas a demonstração da 1.4, já que tem o mesmo efeito da função no pipe.

In [ ]:
from langchain_core.runnables import RunnableLambda

sequencia = RunnableLambda(mascarar) | RunnableLambda(preprocessar)

print("@chain          ->", type(preprocessar_chain).__name__)
print("função no pipe  ->", type(RunnableLambda(preprocessar)).__name__)
print("composição  A|B ->", type(sequencia).__name__)
print("pipeline completo ->", type(pipeline).__name__)

## 1.12 Itens da rubrica → célula

| # | Item da rubrica (Fase 1) | Onde está | O que mostrar |
|---|---|---|---|
| 1 | Anatomia, tipo e maturidade agêntica | **1.1** | 7 etapas numeradas, tipo híbrido, maturidade 0→4 (hoje nível 1) |
| 2 | Planejamento e memória aumentada | **1.2** | Planejamento hierárquico em 5 etapas e as 4 memórias CoALA |
| 3 | Templates reutilizáveis (`PromptTemplate`, `ChatPromptTemplate`) | **1.3** | `inspect.getsource` de `agente/prompts.py` e `format` de exemplo |
| 4 | Saída estruturada em JSON com output parser | **1.5** (e saída de **1.7**) | `JsonOutputParser` + `AnaliseIncidente`, validação e `invoke` |
| 5 | Interface Runnable (`invoke`, `batch`, `stream`) e `@chain` | **1.7**, **1.8**, **1.9**; `@chain` em **1.4** | Saídas das três chamadas e comparação `True` do `@chain` |
| 6 | LCEL e justificativa | **1.6** (chain) e **1.11** (texto) | `montar_pipeline` com `inspect.getsource` e comparação declarativa × imperativa |

Complementos: máscara e pré-processamento (**1.4**), bateria de rotas e verificação de segredos (**1.10**).